# MVTec Anomaly Detection — All Categories
**Pipeline:** Preprocessing → Harris → Pyramid → SIFT → Segmentation → Classification

Select a category from the Gradio interface and everything runs automatically.

**Dataset:** `/kaggle/input/mvtec-ad/`

## 0. Setup & Imports

In [5]:
import os
import glob
import numpy as np

# ── Matplotlib backend لازم يتعمل قبل أي import تاني ─────────────────────
import matplotlib
matplotlib.use('Agg')          # بدون GUI — مهم جداً في Kaggle + Gradio
import matplotlib.pyplot as plt

import cv2
import io
from PIL import Image
import gradio as gr

from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import copy
import warnings
warnings.filterwarnings('ignore')

# ── Paths & Config ────────────────────────────────────────────────────────
# عدّلي المسار لو الداتاست في مكان تاني
DATASET_ROOT = '/kaggle/input/datasets/ipythonx/mvtec-ad'

ALL_CATEGORIES = [
    'bottle',
    'zipper',
    'toothbrush',
    'capsule',
]

IMG_SIZE = (256, 256)
SEED     = 42
np.random.seed(SEED)

sift = cv2.SIFT_create(nfeatures=300)

# ── Model Checkpoint Directory ────────────────────────────────────────────
import joblib
MODEL_DIR = '/kaggle/working/model_checkpoints'
os.makedirs(MODEL_DIR, exist_ok=True)

def model_path(category):
    """Return the checkpoint file path for a given category."""
    return os.path.join(MODEL_DIR, f'{category}_checkpoint.pkl')

# ── Global state for Gradio ───────────────────────────────────────────────
_state = {
    'category'     : None,
    'clf'          : None,
    'scaler'       : None,
    'loc_clf'      : None,  # Supervised pixel-level classifier for bounding boxes
    'pixel_scaler' : None,  # Scaler for pixel features
    'results'      : None,
    'best_name'    : None,
    'defects'      : [],
}

print('✓ Imports & Setup done')

✓ Imports & Setup done


## 1. Core Pipeline Functions

In [6]:
# ─────────────────────────────────────────────────────────────────────────
# DATA LOADING (Updated to load Ground Truth masks)
# ─────────────────────────────────────────────────────────────────────────
def load_images(folder, label, gt_folder=None):
    paths = sorted(glob.glob(os.path.join(folder, '*.png')))
    imgs, labels, masks = [], [], []
    
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        imgs.append(img)
        labels.append(label)
        
        # Load associated Ground Truth Mask if applicable
        mask = np.zeros((IMG_SIZE[1], IMG_SIZE[0]), dtype=np.uint8)
        if gt_folder and label == 1:
            basename = os.path.basename(p)
            name, ext = os.path.splitext(basename)
            mask_path = os.path.join(gt_folder, name + '_mask' + ext)
            if os.path.exists(mask_path):
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is not None:
                    mask = cv2.resize(m, IMG_SIZE)
                    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        masks.append(mask)
        
    return imgs, labels, masks

def load_category(category):
    base       = os.path.join(DATASET_ROOT, category)
    train_good = os.path.join(base, 'train', 'good')
    test_dir   = os.path.join(base, 'test')
    gt_dir     = os.path.join(base, 'ground_truth')

    if not os.path.isdir(train_good):
        raise FileNotFoundError(f'Training path not found: {train_good}')

    # Returns 3 items
    train_imgs, train_labels, train_masks = load_images(train_good, label=0)

    test_imgs, test_labels, test_masks, defect_types = [], [], [], []
    if os.path.isdir(test_dir):
        defect_types = sorted([
            d for d in os.listdir(test_dir)
            if os.path.isdir(os.path.join(test_dir, d))
        ])
        for dt in defect_types:
            lbl  = 0 if dt == 'good' else 1
            gt_folder = os.path.join(gt_dir, dt) if lbl == 1 else None
            # Returns 3 items
            imgs, lbls, msks = load_images(os.path.join(test_dir, dt), label=lbl, gt_folder=gt_folder)
            test_imgs   += imgs
            test_labels += lbls
            test_masks  += msks

    # Returns exactly 7 items
    return train_imgs, train_labels, train_masks, test_imgs, test_labels, test_masks, defect_types


# ─────────────────────────────────────────────────────────────────────────
# PREPROCESSING & FEATURES
# ─────────────────────────────────────────────────────────────────────────
def harris_corners(img_rgb, threshold=0.01):
    gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    response = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)
    response = cv2.dilate(response, None)
    marked   = img_rgb.copy()
    marked[response > threshold * response.max()] = [255, 0, 0]
    count    = int(np.sum(response > threshold * response.max()))
    return marked, count


def kmeans_segment(img_rgb, k=3):
    pixels   = img_rgb.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
    )
    centers  = centers.astype(np.uint8)
    seg_img  = centers[labels.flatten()].reshape(img_rgb.shape)
    lbl_mask = labels.reshape(img_rgb.shape[:2])
    return seg_img, lbl_mask


def extract_features(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gray_u8 = gray.astype(np.uint8)

    gauss  = cv2.GaussianBlur(gray, (5, 5), 1.5)
    median = cv2.medianBlur(gray_u8, 5).astype(np.float32)
    diff_g = np.abs(gray - gauss)
    diff_m = np.abs(gray - median)

    response       = cv2.cornerHarris(gray, 2, 3, 0.04)
    corner_density = float(np.sum(response > 0.01 * response.max())) / (IMG_SIZE[0] * IMG_SIZE[1])

    kps, descs = sift.detectAndCompute(gray_u8, None)
    n_kps      = len(kps)
    resp_mean  = float(np.mean([k.response for k in kps])) if kps else 0.0
    resp_std   = float(np.std ([k.response for k in kps])) if kps else 0.0
    desc_mean  = float(descs.mean()) if descs is not None else 0.0
    desc_std   = float(descs.std())  if descs is not None else 0.0

    lap      = cv2.Laplacian(gray_u8, cv2.CV_64F)
    lap_mean = float(np.mean(np.abs(lap)))
    lap_std  = float(np.std(lap))
    lap_var  = float(lap.var())

    r_mean, g_mean, b_mean = img_rgb[:, :, 0].mean(), img_rgb[:, :, 1].mean(), img_rgb[:, :, 2].mean()
    r_std,  g_std,  b_std  = img_rgb[:, :, 0].std(),  img_rgb[:, :, 1].std(),  img_rgb[:, :, 2].std()

    _, lbl     = kmeans_segment(img_rgb, k=4)
    counts     = np.bincount(lbl.flatten(), minlength=4)[:3]
    seg_ratios = sorted([c / lbl.size for c in counts])

    return np.array([
        diff_g.mean(), diff_g.std(), diff_m.mean(), diff_m.std(),    
        corner_density,                                              
        n_kps, resp_mean, resp_std, desc_mean, desc_std,             
        lap_mean, lap_std, lap_var,                                  
        r_mean, g_mean, b_mean, r_std, g_std, b_std,                 
        *seg_ratios                                                  
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────
# TRAIN PIPELINE 
# ─────────────────────────────────────────────────────────────────────────
def train_pipeline(category):
    train_imgs, train_labels, train_masks, test_imgs, test_labels, test_masks, defect_types = load_category(category)

    if len(train_imgs) == 0:
        raise ValueError(f'No images in train/good for {category}')

    all_imgs   = train_imgs + test_imgs
    all_labels = train_labels + test_labels
    all_masks  = train_masks + test_masks

    X = np.array([extract_features(img) for img in all_imgs])
    y = np.array(all_labels)

    stratify_y = y if len(np.unique(y)) > 1 else None

    # Sync split for Images AND ground_truth masks
    X_tr, X_te, y_tr, y_te, img_tr, img_te, mask_tr, mask_te = train_test_split(
        X, y, all_imgs, all_masks, test_size=0.25, random_state=SEED, stratify=stratify_y
    )

    # 1. Main Global Classification Model
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_te_s   = scaler.transform(X_te)

    classifiers = {
        'Naive Bayes'      : GaussianNB(),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
        'Random Forest'    : RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=SEED),
    }

    results = {}
    trained_clfs = {}
    for name, clf in classifiers.items():
        clf.fit(X_tr_s, y_tr)
        y_pred = clf.predict(X_te_s)
        results[name] = {
            'acc' : accuracy_score(y_te, y_pred),
            'prec': precision_score(y_te, y_pred, zero_division=0),
            'rec' : recall_score(y_te, y_pred, zero_division=0),
            'f1'  : f1_score(y_te, y_pred, zero_division=0),
            'cm'  : confusion_matrix(y_te, y_pred),
        }
        trained_clfs[name] = clf

    best_name = max(results, key=lambda k: results[k]['f1'])
    
    best_clf = copy.deepcopy(trained_clfs[best_name].__class__(
        **trained_clfs[best_name].get_params()
    ))
    X_all_s = scaler.fit_transform(X)
    best_clf.fit(X_all_s, y)

    # 2. Train Localization Model using Ground Truth masks (Supervised Bounding Boxes)
    pixel_X, pixel_y = [], []
    for img, mask, label in zip(img_tr, mask_tr, y_tr):
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (21, 21), 0)
        diff = np.abs(gray.astype(np.float32) - blur.astype(np.float32))
        lap = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
        
        # Pixel-level features
        feats = np.dstack([img, diff, lap]).reshape(-1, 5) 
        flat_mask = mask.flatten()

        if label == 1 and flat_mask.max() > 0:
            def_idx = np.where(flat_mask > 0)[0]
            norm_idx = np.where(flat_mask == 0)[0]
            np.random.shuffle(def_idx)
            np.random.shuffle(norm_idx)
            sel_def = def_idx[:800] # Train on defect pixels
            sel_norm = norm_idx[:2000] # HEAVY training on background to prevent false positives
            
            pixel_X.extend([feats[sel_def], feats[sel_norm]])
            pixel_y.extend([np.ones(len(sel_def)), np.zeros(len(sel_norm))])
        elif label == 0:
            norm_idx = np.where(flat_mask == 0)[0]
            np.random.shuffle(norm_idx)
            sel_norm = norm_idx[:500] 
            pixel_X.append(feats[sel_norm])
            pixel_y.append(np.zeros(len(sel_norm)))

    if pixel_X:
        pixel_X = np.vstack(pixel_X)
        pixel_y = np.concatenate(pixel_y)
        pixel_scaler = StandardScaler()
        pixel_X_s = pixel_scaler.fit_transform(pixel_X)
        loc_clf = RandomForestClassifier(n_estimators=30, max_depth=10, random_state=SEED)
        loc_clf.fit(pixel_X_s, pixel_y)
    else:
        loc_clf = None
        pixel_scaler = None

    # Returns exactly 7 variables
    return best_clf, scaler, results, best_name, defect_types, loc_clf, pixel_scaler


# ─────────────────────────────────────────────────────────────────────────
# GRADIO HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────
def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img

def train_on_category(category):
    if not category: return 'Please select a category first!', None, None
    try:
        chk = model_path(category)
        if os.path.exists(chk):
            # ── Load saved checkpoint instead of retraining ──────────────
            saved = joblib.load(chk)
            clf          = saved['clf']
            scaler       = saved['scaler']
            results      = saved['results']
            best_name    = saved['best_name']
            defects      = saved['defects']
            loc_clf      = saved['loc_clf']
            pixel_scaler = saved['pixel_scaler']
            loaded_from_disk = True
        else:
            # ── Train from scratch and save checkpoint ───────────────────
            clf, scaler, results, best_name, defects, loc_clf, pixel_scaler = train_pipeline(category)
            joblib.dump({
                'clf': clf, 'scaler': scaler, 'results': results,
                'best_name': best_name, 'defects': defects,
                'loc_clf': loc_clf, 'pixel_scaler': pixel_scaler,
            }, chk)
            loaded_from_disk = False
    except Exception as e: 
        import traceback
        return f'Error during training:\n{traceback.format_exc()}', None, None

    # Save to global state
    _state.update({
        'category': category, 'clf': clf, 'scaler': scaler, 
        'loc_clf': loc_clf, 'pixel_scaler': pixel_scaler,
        'results': results, 'best_name': best_name, 'defects': defects
    })

    fig1, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig1.suptitle(f'Classifier Performance — {category}', fontsize=13, fontweight='bold')
    colors  = ['#4C72B0', '#DD8452', '#55A868']
    metrics = ['acc', 'prec', 'rec', 'f1']
    for ax, (cname, color) in zip(axes, zip(results.keys(), colors)):
        r = results[cname]; vals = [r[m] for m in metrics]
        bars = ax.bar(['Accuracy', 'Precision', 'Recall', 'F1'], vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_ylim(0, 1.12); ax.set_title(cname, fontsize=9, fontweight='bold')
        for bar, val in zip(bars, vals): ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout(); metrics_img = fig_to_pil(fig1)

    fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
    fig2.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
    for ax, (cname, color) in zip(axes2, zip(results.keys(), colors)):
        cm = results[cname]['cm']; ax.imshow(cm, cmap='Blues'); ax.set_title(cname, fontsize=9)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(['Normal', 'Defect']); ax.set_yticklabels(['Normal', 'Defect'])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
    plt.tight_layout(); cm_img = fig_to_pil(fig2)

    r = results[best_name]
    source_label = f'✓ Loaded saved checkpoint: {chk}' if loaded_from_disk else f'✓ Model trained & saved to: {chk}'
    status = (
        f'{source_label}\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'Defect types  : {defects}\n\nBest model    : {best_name}\n'
        f'  Accuracy    : {r["acc"]:.4f}\n  Precision   : {r["prec"]:.4f}\n'
        f'  Recall      : {r["rec"]:.4f}\n  F1 Score    : {r["f1"]:.4f}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nNow upload an image and click Analyze!'
    )
    return status, metrics_img, cm_img


def analyze_image(pil_image):
    if pil_image is None: return None, 'Please upload an image first!'
    if _state['clf'] is None: return None, 'Please train the model first!'

    try:
        img_rgb = np.array(pil_image.convert('RGB')); img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        feat = extract_features(img_rgb).reshape(1, -1); feat_s = _state['scaler'].transform(feat)
        pred = _state['clf'].predict(feat_s)[0]
        proba = _state['clf'].predict_proba(feat_s)[0] if hasattr(_state['clf'], 'predict_proba') else None

        label_str = 'NORMAL ✓' if pred == 0 else 'DEFECTIVE ✗'; color_hex = '#2d8a4e' if pred == 0 else '#c0392b'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        seg, _ = kmeans_segment(img_rgb, k=3); harris_m, cnt = harris_corners(img_rgb)
        gray_u8 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY); kps_img, _ = sift.detectAndCompute(gray_u8, None)
        sift_drawn = cv2.drawKeypoints(gray_u8, kps_img, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4)); fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(f'Category: {_state["category"]}  |  {label_str}  |  Confidence: {confidence}', fontsize=13, fontweight='bold', color=color_hex, y=1.03)
        axes[0].imshow(img_rgb); axes[0].set_title('Input Image'); axes[0].axis('off')
        axes[1].imshow(seg); axes[1].set_title('K-Means Segmentation (K=3)'); axes[1].axis('off')
        axes[2].imshow(harris_m); axes[2].set_title(f'Harris Corners ({cnt})'); axes[2].axis('off')
        axes[3].imshow(sift_drawn, cmap='gray'); axes[3].set_title(f'SIFT Keypoints ({len(kps_img)})'); axes[3].axis('off')
        plt.tight_layout(); result_img = fig_to_pil(fig)

        proba_str = f'  P(Normal)    = {proba[0]*100:.1f}%\n  P(Defective) = {proba[1]*100:.1f}%\n' if proba is not None else ''
        report = (
            f'============================================\n  ANOMALY DETECTION REPORT\n============================================\n'
            f'  Category     : {_state["category"]}\n  Prediction   : {label_str}\n  Confidence   : {confidence}\n{proba_str}'
            f'  Harris corners : {cnt}\n  SIFT keypoints : {len(kps_img)}\n  Best model     : {_state["best_name"]}\n============================================'
        )
        return result_img, report
    except Exception as e: return None, f'Error during analysis:\n{e}'


def detect_defects(pil_image):
    if pil_image is None: return None, 'Please upload an image first!'
    if _state['clf'] is None: return None, 'Please train the model first!'

    try:
        img_rgb = np.array(pil_image.convert('RGB'))
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        gray_u8 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        gray_f = gray_u8.astype(np.float32)
        H, W = IMG_SIZE[1], IMG_SIZE[0]

        # 1. UNIFIED CLASSIFICATION
        feat = extract_features(img_rgb).reshape(1, -1)
        feat_s = _state['scaler'].transform(feat)
        pred = _state['clf'].predict(feat_s)[0]
        proba = _state['clf'].predict_proba(feat_s)[0] if hasattr(_state['clf'], 'predict_proba') else None

        label_str = 'DEFECTIVE ✗' if pred == 1 else 'NORMAL ✓'
        color_hex = '#c0392b' if pred == 1 else '#2d8a4e'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        # 2. Generate Supervised Anomaly Map (Predict Pixels)
        score_map = np.zeros(IMG_SIZE, dtype=np.float32)
        if _state['loc_clf'] is not None and _state['pixel_scaler'] is not None:
            bg_blur = cv2.GaussianBlur(gray_f, (21, 21), 0)
            diff = np.abs(gray_f - bg_blur)
            lap = np.abs(cv2.Laplacian(gray_u8, cv2.CV_32F, ksize=3))
            
            px_feats = np.dstack([img_rgb, diff, lap]).reshape(-1, 5)
            px_feats_s = _state['pixel_scaler'].transform(px_feats)
            
            # Predict bounding box annotation via pixel probability
            px_probs = _state['loc_clf'].predict_proba(px_feats_s)[:, 1]
            score_map = px_probs.reshape(IMG_SIZE[1], IMG_SIZE[0])
            
            # Smooth out the noise (prevents false positive boxes)
            score_map = cv2.GaussianBlur(score_map, (9, 9), 0)
            
        max_val = score_map.max()
        score_u8 = np.clip(score_map * (255.0 / max_val), 0, 255).astype(np.uint8) if max_val > 0 else np.zeros_like(gray_u8)

        heatmap = cv2.cvtColor(cv2.applyColorMap(score_u8, cv2.COLORMAP_INFERNO), cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_rgb, 0.50, heatmap, 0.50, 0)

        annotated = img_rgb.copy()
        defect_regions = []

        # Ensure all coordinates are clamped inside the image limits
        def draw_box_clamped(img, x, y, w, h, color=(220, 30, 30), thickness=2):
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        # 3. DRAW DYNAMIC BOUNDING BOXES 
        if pred == 1 and max_val >= 0.35: 
            
            # Relaxed threshold (45%) captures the faded outer edges of defects
            _, binary = cv2.threshold(score_map, 0.45, 255, cv2.THRESH_BINARY)
            binary = binary.astype(np.uint8)
            
            # Heavier morphological inflation to guarantee 100% of the defect area is boxed
            binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
            binary = cv2.dilate(binary, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
            
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            min_area = W * H * 0.0005 
            max_area = W * H * 0.60 # Reject boxes over 60% of the image

            scored = []
            for cnt_c in contours:
                c_area = cv2.contourArea(cnt_c)
                if min_area <= c_area <= max_area:
                    mask_c = np.zeros(score_map.shape, np.uint8)
                    cv2.drawContours(mask_c, [cnt_c], -1, 255, -1)
                    rel_score = float(score_map[mask_c > 0].max()) / max_val
                    scored.append((rel_score, cnt_c))
            
            scored.sort(key=lambda x: x[0], reverse=True)
            
            # Allow up to 15 boxes for scattered defects (like drops or multiple scratches)
            for region_score, cnt_c in scored[:15]:
                x, y, w, h = cv2.boundingRect(cnt_c)
                
                # Double-check bounding box area doesn't exceed limit
                if (w * h) <= max_area:
                    defect_regions.append((x, y, w, h, region_score))
                    box_color = (220, 30, 30) 
                    
                    draw_box_clamped(annotated, x, y, w, h, color=box_color, thickness=2)
                    
                    label_txt = f'DEFECT'
                    (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
                    
                    # Clamp text so it never draws above y=0
                    lx, ly = max(0, x), max(th + 2, y - 4)
                    
                    cv2.rectangle(annotated, (lx, ly - th - 2), (lx + tw + 4, ly + 2), box_color, -1)
                    cv2.putText(annotated, label_txt, (lx + 2, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

            banner = f'{label_str}  ({len(defect_regions)} region(s))'
            cv2.rectangle(annotated, (0, 0), (len(banner)*9 + 8, 28), (220, 30, 30), -1)
            cv2.putText(annotated, banner, (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            cv2.rectangle(annotated, (0, 0), (len(label_str)*9 + 8, 28), (45, 138, 78), -1)
            cv2.putText(annotated, label_str, (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

        # Plotting
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(f'Defect Detection  |  Category: {_state["category"]}  |  {label_str}', fontsize=12, fontweight='bold', color=color_hex)
        axes[0].imshow(img_rgb); axes[0].set_title('Input Image'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('Supervised Annotation Map'); axes[1].axis('off')
        axes[2].imshow(annotated); axes[2].set_title('Localized Defect Areas'); axes[2].axis('off')

        plt.colorbar(plt.cm.ScalarMappable(cmap='inferno', norm=plt.Normalize(0, 1)), ax=axes[1], fraction=0.046, pad=0.04).set_label('Annotation Match')
        plt.tight_layout()
        result_pil = fig_to_pil(fig)

        proba_str = f'  P(Normal)      = {proba[0]*100:.1f}%\n  P(Defective)   = {proba[1]*100:.1f}%\n' if proba is not None else ''
        region_str = f'  Defect regions : {len(defect_regions)} area(s) detected\n' + ''.join([f'    Region {idx}: x={x}, y={y}, w={w}, h={h}\n' for idx, (x, y, w, h, rs) in enumerate(defect_regions, 1)]) if defect_regions else '  Defect regions : None detected\n'

        report = (
            f'============================================\n  DEFECT DETECTION REPORT\n============================================\n'
            f'  Category       : {_state["category"]}\n  Verdict        : {label_str}\n  Confidence     : {confidence}\n{proba_str}'
            f'  Peak Intensity : {max_val:.1f}\n{region_str}'
            f'  Model used     : {_state["best_name"]} & Ground Truth Localization\n============================================'
        )
        return result_pil, report

    except Exception as e: 
        import traceback
        return None, f'Error during defect detection:\n{traceback.format_exc()}'

print('✓ Core logic and helper functions loaded')

✓ Core logic and helper functions loaded


## 2. Gradio Interface

In [7]:
!pip install gradio -q

In [8]:
# ─────────────────────────────────────────────────────────────────────────
# GRADIO INTERFACE LAYOUT
# ─────────────────────────────────────────────────────────────────────────
with gr.Blocks(title='MVTec AD — Multi-Category Anomaly Detector', theme=gr.themes.Soft()) as demo:
    gr.Markdown("# MVTec Anomaly Detection — All Categories\n**Steps:**\n1. Select a category\n2. Click **Train Model**\n3. Go to **Analyze Image** or **Defect Detection**")

    with gr.Row():
        with gr.Column(scale=2):
            cat_dropdown = gr.Dropdown(choices=ALL_CATEGORIES, label='Select Category', value='bottle')
            train_btn = gr.Button('Train Model', variant='primary', size='lg')
        with gr.Column(scale=3):
            train_status = gr.Textbox(label='Training Status', lines=10, placeholder='Select a category and click Train...')

    with gr.Row():
        metrics_out = gr.Image(label='Classifier Metrics')
        cm_out      = gr.Image(label='Confusion Matrices')

    gr.Markdown('---')

    with gr.Tabs():
        with gr.TabItem('Analyze Image'):
            gr.Markdown('### Upload an image to run the full analysis pipeline')
            with gr.Row():
                with gr.Column(scale=1):
                    inp_img     = gr.Image(type='pil', label='Upload image from test/', height=280)
                    analyze_btn = gr.Button('Analyze Image', variant='secondary', size='lg')
                with gr.Column(scale=2):
                    out_vis  = gr.Image(label='Visual Analysis', height=350)
                    out_text = gr.Textbox(label='Detection Report', lines=12, show_copy_button=True)

        with gr.TabItem('Defect Detection'):
            gr.Markdown("### Defect Detection — Localize anomalous regions\nUpload an image and the model will:\n- Classify it as **Normal** or **Defective**\n- Map to learned **ground truth masks**\n- Draw **red bounding boxes** around defects")
            with gr.Row():
                with gr.Column(scale=1):
                    dd_img = gr.Image(type='pil', label='Upload image', height=280)
                    dd_btn = gr.Button('Detect Defects', variant='primary', size='lg')
                with gr.Column(scale=2):
                    dd_vis    = gr.Image(label='Defect Detection Result', height=380)
                    dd_report = gr.Textbox(label='Defect Detection Report', lines=14, show_copy_button=True)

    # Event wiring
    train_btn.click(fn=train_on_category, inputs=[cat_dropdown], outputs=[train_status, metrics_out, cm_out])
    analyze_btn.click(fn=analyze_image, inputs=[inp_img], outputs=[out_vis, out_text])
    dd_btn.click(fn=detect_defects, inputs=[dd_img], outputs=[dd_vis, dd_report])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://7429561bbebccc685b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error